In [ ]:
import pandas as pd
import numpy as np

# analysis window
START_DATE = pd.Timestamp("2023-12-29")
END_DATE   = pd.Timestamp("2025-12-31")
sec_px_csv = "sec_px.csv" # path to security price data
sec_metadata_csv = "sec_metadata.csv" # path to security metadata


In [ ]:
def load_sec_px(path: str) -> pd.DataFrame:
    px = pd.read_csv(
        path,
        parse_dates=["date"],
        dayfirst=False
    )
    px = px.set_index("date").sort_index()
    px = px.apply(pd.to_numeric, errors="coerce")
    return px
sec_px = load_sec_px(sec_px_csv)

def restrict_window(px: pd.DataFrame,
                    start: pd.Timestamp,
                    end: pd.Timestamp) -> pd.DataFrame:
    return px.loc[(px.index >= start) & (px.index <= end)]
sec_px_window = restrict_window(sec_px, START_DATE, END_DATE)

def filter_eligible_securities(px: pd.DataFrame) -> pd.DataFrame:
    start_prices = px.iloc[0]
    end_prices   = px.iloc[-1]

    eligible = start_prices.notna() & end_prices.notna()
    px = px.loc[:, eligible]

    returns = px.pct_change()
    has_returns = returns.notna().sum() > 0

    return px.loc[:, has_returns]
sec_px_clean = filter_eligible_securities(sec_px_window)

In [ ]:
def load_sec_metadata(path: str) -> pd.DataFrame:
    meta = pd.read_csv(path)
    meta.columns = meta.columns.str.lower()
    meta["date"] = pd.to_datetime(meta["date"])
    return meta
sec_metadata = load_sec_metadata(sec_metadata_csv)
def build_name_map(meta: pd.DataFrame) -> pd.Series:
    latest = (
        meta.sort_values("date")
            .groupby("sedol")
            .tail(1)
    )
    return latest.set_index("sedol")["name"]

name_map = build_name_map(sec_metadata)
name_map.head()

sedol
TS150329                               Far East Horizon (Det)
BD5CK8      CETC Cyberspace Security Technology Co. Ltd. C...
BD5CK3      Yuan Longping High-Tech Agriculture Co., Ltd. ...
BD5CKR                            SeeWay.ai Co., Ltd. Class A
BD5LY1                 Apeloa Pharmaceutical Co. Ltd. Class A
Name: name, dtype: object

In [ ]:
def compute_annualised_twr(px: pd.DataFrame) -> pd.DataFrame:
    """
    Compute annualised Time-Weighted Return (TWR) for each security
    using monthly price data.

    Parameters
    ----------
    px : pd.DataFrame
        Cleaned price panel with DatetimeIndex and SEDOL columns

    Returns
    -------
    pd.DataFrame
        Columns: sedol, annualised_twr, n_months
    """
    # Monthly returns
    returns = px.pct_change()

    results = []

    for sedol in returns.columns:
        r = returns[sedol].dropna()

        if len(r) == 0:
            continue

        # Total time-weighted return
        twr_total = (1 + r).prod() - 1

        # Annualise based on number of months observed
        annualised_twr = (1 + twr_total) ** (12 / len(r)) - 1

        results.append({
            "sedol": sedol,
            "annualised_twr": annualised_twr,
            "n_months": len(r)
        })

    return pd.DataFrame(results)

twr_df = compute_annualised_twr(sec_px_clean)


25


In [37]:
twr_df["annualised_twr"].describe()
# twr_df["n_months"].describe()

# twr_df["n_months"].min(), twr_df["n_months"].max()

count    814.000000
mean       0.180755
std        0.332745
min       -0.436901
25%       -0.019264
50%        0.118526
75%        0.312294
max        2.946943
Name: annualised_twr, dtype: float64

In [38]:
def attach_security_names(twr_df: pd.DataFrame,
                          name_map: pd.Series) -> pd.DataFrame:
    out = twr_df.copy()
    out["name"] = out["sedol"].map(name_map)
    return out
twr_df_named = attach_security_names(twr_df, name_map)

In [ ]:
# Get final result
def get_top_bottom_securities(twr_named: pd.DataFrame,
                              n: int = 3) -> tuple[pd.DataFrame, pd.DataFrame]:
    ranked = twr_named.dropna(subset=["name"])

    top = (
        ranked.sort_values("annualised_twr", ascending=False)
              .head(n)
              .loc[:, ["name", "annualised_twr"]]
    )

    bottom = (
        ranked.sort_values("annualised_twr", ascending=True)
              .head(n)
              .loc[:, ["name", "annualised_twr"]]
    )

    return top, bottom
top_securities, bottom_securities = get_top_bottom_securities(twr_df_named)
print("Top 3 Securities:")
print(top_securities)   
print("\nBottom 3 Securities:")
print(bottom_securities)

Top 3 Securities:
                                                  name  annualised_twr
325  Victory Giant Technology (HuiZhou) Co., Ltd. C...        2.946943
326            Eoptolink Technology Inc., Ltd. Class A        2.488145
504          Cambricon Technologies Corp. Ltd. Class A        2.152435

Bottom 3 Securities:
                                                  name  annualised_twr
198  Chongqing Zhifei Biological Products Co., Ltd....       -0.436901
484               Hygeia Healthcare Holdings Co., Ltd.       -0.413385
7                   Legend Biotech Corp. Sponsored ADR       -0.406499


In [ ]:
def export_results(top: pd.DataFrame, bottom: pd.DataFrame,
                   filepath: str = "part_a_results.xlsx") -> None:

    # Prepare display-ready dataframes
    def format_df(df, label_start):
        out = df[["name", "annualised_twr"]].copy()
        out.columns = ["Security Name", "Annualised TWR"]
        out.insert(0, "Rank", range(1, len(out) + 1))
        out["Annualised TWR"] = (out["Annualised TWR"] * 100).round(2).astype(str) + "%"
        out.insert(0, "Category", label_start)
        return out

    top_fmt    = format_df(top, "Top 3")
    bottom_fmt = format_df(bottom, "Bottom 3")
    combined   = pd.concat([top_fmt, bottom_fmt], ignore_index=True)

    if filepath.endswith(".xlsx"):
        with pd.ExcelWriter(filepath, engine="openpyxl") as writer:
            combined.to_excel(writer, sheet_name="TWR Results", index=False)

            # Auto-fit column widths
            ws = writer.sheets["TWR Results"]
            for col in ws.columns:
                max_len = max(len(str(cell.value)) for cell in col if cell.value)
                ws.column_dimensions[col[0].column_letter].width = max_len + 4

        print(f"Excel saved → {filepath}")

    else: 
        combined.to_csv(filepath, index=False)
        print(f"CSV saved → {filepath}")

export_results(top_securities, bottom_securities)
